# 03 -- Audio avec pydub

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- charger et sauvegarder des fichiers audio (WAV, MP3, OGG) ;
- manipuler la duree, le volume, la vitesse ;
- decouper, concatener, overlayer des segments ;
- appliquer du fade in/out ;
- detecter les silences ;
- exporter vers differents formats ;
- generer des tonalites simples.

## Prerequis -- ce que vous devez maitriser

- les bases Python (fonctions, slicing) ;
- installation de ffmpeg (requis par pydub pour MP3/OGG).

Ce que ce notebook **ne couvrira pas** :

- le traitement signal en detail (voir `Data_science/05_SciPy.ipynb`) ;
- librosa / analyse musicale avancee ;
- la synthese vocale.

## Plan

1. Installation
2. Charger un fichier audio
3. Proprietes
4. Volume et gain
5. Decoupage
6. Concatenation
7. Overlay et mixing
8. Fade in/out
9. Reverse et vitesse
10. Silences
11. Generer des tonalites
12. Exporter
13. Synthese
14. Exercices

---

## 1. Installation

In [ ]:
# pip install pydub
# sudo apt install ffmpeg (ou brew install ffmpeg)
from pydub import AudioSegment
from pydub.generators import Sine, WhiteNoise, Square
import pydub
pydub.__version__ if hasattr(pydub, '__version__') else 'installed'

---

## 2. Charger un fichier audio

In [ ]:
# sound = AudioSegment.from_file('demo.mp3')
# sound = AudioSegment.from_wav('demo.wav')
# sound = AudioSegment.from_mp3('demo.mp3')
print('from_file detecte le format via ffmpeg')

---

## 3. Proprietes

In [ ]:
# Creer un sample synthetique pour la demo
sine = Sine(440).to_audio_segment(duration=1000)  # 1 seconde de La 440Hz
print('duree   :', len(sine), 'ms')
print('canaux  :', sine.channels)
print('sample  :', sine.frame_rate, 'Hz')
print('width   :', sine.sample_width, 'octets')

---

## 4. Volume et gain

In [ ]:
louder = sine + 6  # +6 dB
quieter = sine - 10  # -10 dB
print('dBFS sine :', sine.dBFS)
print('dBFS louder :', louder.dBFS)

---

## 5. Decoupage

In [ ]:
# Slicing en millisecondes
first_half = sine[:500]
last_half = sine[500:]
print(len(first_half), len(last_half))

---

## 6. Concatenation

In [ ]:
tone_a = Sine(440).to_audio_segment(duration=500)
tone_b = Sine(880).to_audio_segment(duration=500)
combined = tone_a + tone_b
print('duree combinee :', len(combined), 'ms')

---

## 7. Overlay et mixing

In [ ]:
bg = Sine(220).to_audio_segment(duration=2000).apply_gain(-20)
voice = Sine(440).to_audio_segment(duration=1000)
mix = bg.overlay(voice, position=500)
print(len(mix), 'ms')

---

## 8. Fade in/out

In [ ]:
tone = Sine(440).to_audio_segment(duration=2000)
faded = tone.fade_in(500).fade_out(500)
print('fade in/out appliques')

---

## 9. Reverse et vitesse

In [ ]:
reversed_tone = tone.reverse()
print('reverse applique')

In [ ]:
# Changer la vitesse en modifiant le frame_rate
def change_speed(sound: AudioSegment, factor: float) -> AudioSegment:
    sound_fast = sound._spawn(sound.raw_data, overrides={'frame_rate': int(sound.frame_rate * factor)})
    return sound_fast.set_frame_rate(sound.frame_rate)

fast = change_speed(tone, 1.5)
print(len(fast), 'ms')

---

## 10. Silences

In [ ]:
from pydub.silence import detect_silence, split_on_silence

# Construire un sample avec silence au milieu
audio = Sine(440).to_audio_segment(duration=500) + AudioSegment.silent(duration=300) + Sine(880).to_audio_segment(duration=500)
silences = detect_silence(audio, min_silence_len=100, silence_thresh=-40)
print('Silences detectes :', silences)

In [ ]:
# split sur les silences
chunks = split_on_silence(audio, min_silence_len=100, silence_thresh=-40)
len(chunks)

---

## 11. Generer des tonalites

In [ ]:
la = Sine(440).to_audio_segment(duration=500)
la_sharp = Sine(466.16).to_audio_segment(duration=500)
si = Sine(493.88).to_audio_segment(duration=500)
melody = la + la_sharp + si
print('Melodie de 3 notes')

In [ ]:
noise = WhiteNoise().to_audio_segment(duration=500).apply_gain(-20)
square = Square(200).to_audio_segment(duration=500)
print('WhiteNoise et Square disponibles')

---

## 12. Exporter

In [ ]:
import tempfile, pathlib
tmp = pathlib.Path(tempfile.gettempdir())
tone.export(str(tmp / 'tone.wav'), format='wav')
# tone.export(str(tmp / 'tone.mp3'), format='mp3', bitrate='192k')
print('Exporte en', tmp)

---

## Synthese

| Operation | pydub |
|---|---|
| Charger | `AudioSegment.from_file(path)` |
| Duree | `len(sound)` (ms) |
| Volume | `sound + dB` / `sound - dB` |
| Slicing | `sound[start_ms:end_ms]` |
| Concat | `a + b` |
| Overlay | `a.overlay(b, position=ms)` |
| Fade | `.fade_in(ms)`, `.fade_out(ms)` |
| Reverse | `.reverse()` |
| Silences | `detect_silence`, `split_on_silence` |
| Tonalite | `Sine(freq).to_audio_segment(duration=ms)` |
| Export | `.export(path, format='mp3')` |

### Regles a retenir

- **Durees en millisecondes**, pas secondes.
- **Volume en dB** : `+6` = double amplitude.
- **ffmpeg requis** pour les formats autres que WAV.
- **`AudioSegment` immutable** : chaque operation cree un nouveau segment.

---

## Exercices

### Exercice 1 (facile) -- 5 secondes de La 440Hz

Generez 5 secondes de sine 440Hz et exportez en WAV.

In [ ]:
from pydub import AudioSegment
from pydub.generators import Sine

# TODO

<details>
<summary>Voir la correction</summary>

```python
from pydub.generators import Sine
import pathlib, tempfile

tone = Sine(440).to_audio_segment(duration=5000)
out = pathlib.Path(tempfile.gettempdir()) / 'la.wav'
tone.export(str(out), format='wav')
print('fichier :', out)
```

</details>

### Exercice 2 (facile) -- Gamme

Creez une gamme montante C-D-E-F-G-A-B-C (frequences 261.63, 293.66, ...).

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from pydub import AudioSegment
from pydub.generators import Sine

notes_hz = [261.63, 293.66, 329.63, 349.23, 392.00, 440.00, 493.88, 523.25]
gamme = AudioSegment.silent(duration=0)
for freq in notes_hz:
    gamme += Sine(freq).to_audio_segment(duration=400)
print('duree :', len(gamme))
```

</details>

### Exercice 3 (moyen) -- Fade et mix

Creez un clip de 3 secondes de sine 440Hz en fade in/out de 500ms, overlay avec un fond de bruit blanc a -20dB.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from pydub.generators import Sine, WhiteNoise

tone = Sine(440).to_audio_segment(duration=3000).fade_in(500).fade_out(500)
bg = WhiteNoise().to_audio_segment(duration=3000).apply_gain(-20)
mix = bg.overlay(tone)
print(len(mix))
```

</details>

### Exercice 4 (moyen) -- Decoupage

Fonction `first_seconds(path, seconds)` qui extrait les N premieres secondes d'un fichier audio.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from pydub import AudioSegment

def first_seconds(path: str, seconds: float) -> AudioSegment:
    sound = AudioSegment.from_file(path)
    return sound[:int(seconds * 1000)]
```

</details>

### Exercice 5 (difficile) -- Retirer les silences

Fonction `trim_silence(path, thresh_db)` qui retire les silences au debut et a la fin.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from pydub import AudioSegment
from pydub.silence import detect_nonsilent

def trim_silence(path: str, thresh_db: float = -40) -> AudioSegment:
    sound = AudioSegment.from_file(path)
    nonsilent = detect_nonsilent(sound, min_silence_len=100, silence_thresh=thresh_db)
    if not nonsilent:
        return sound
    start = nonsilent[0][0]
    end = nonsilent[-1][1]
    return sound[start:end]
```

</details>

---

## Ressources externes

- [pydub GitHub](https://github.com/jiaaro/pydub)
- [pydub API](https://github.com/jiaaro/pydub/blob/master/API.markdown)
- [ffmpeg](https://ffmpeg.org/)